# Extract abstract numbers

Pulls every PROFILE-derivable number that appears in the CAIA abstract. Each section corresponds to one sentence in the abstract.

Site cohort sizes (DFCI / MSKCC / JHU) are external — listed as configured constants so it's clear what's hardcoded vs derived from local artifacts.

## Paths

Edit `RUNS_DIR`, `AGGREGATED`, `V3_LABELS` to point at your data.

In [ ]:
from pathlib import Path
import pandas as pd

RUNS_DIR   = Path.home() / "Desktop" / "local_runs"
AGGREGATED = Path.home() / "Desktop" / "local_runs" / "_inputs" / "aggregated_landmark0.csv"
V3_LABELS  = Path.home() / "Desktop" / "local_runs" / "_inputs" / "LLM_v3_labels.tsv"

# External-site cohort numbers (not derivable from PROFILE). Update as new pulls land.
SITE_COHORTS = {
    "DFCI":  {"n": 7942,  "n_platinum": 176},
    "MSKCC": {"n": 15557, "n_platinum": 98},
    "JHU":   {"n": 4841,  "n_platinum": 25},
}

def fmt_pct(num, denom):
    return f"{num}/{denom} ({100.0 * num / denom:.1f}%)"

for label, p in [("runs_dir", RUNS_DIR), ("aggregated", AGGREGATED), ("v3_labels", V3_LABELS)]:
    print(f"  {label:11s} {p}  ({'OK' if p.exists() else 'MISSING'})")

## 1. PROFILE cohort and platinum initiation

Abstract: *"Of 1,252 PROFILE patients, 129 (10.3%) initiated platinum."*

In [ ]:
agg = pd.read_csv(AGGREGATED)
n_total = len(agg)
n_platinum = int(pd.to_numeric(agg["PLATINUM"], errors="coerce").fillna(0).sum())
n_death = int(pd.to_numeric(agg["DEATH"], errors="coerce").fillna(0).sum())

print(f"source: {AGGREGATED}")
print(f"PROFILE patients .......... {n_total}")
print(f"Initiated platinum ........ {fmt_pct(n_platinum, n_total)}")
print(f"Died ...................... {fmt_pct(n_death, n_total)}")

## 2. NEPC / AVPC prevalence by platinum exposure

Abstract: *"NEPC/AVPC was identified in 86% of platinum-treated patients (NEPC 18%, AVPC 68%) vs 35% of platinum-naïve (NEPC 3%, AVPC 32%)."*

Joins LLM v3 labels (`has_nepc`, `has_avpc`) with PLATINUM status from the aggregated cohort.

In [ ]:
agg_pl = pd.read_csv(AGGREGATED, usecols=["DFCI_MRN", "PLATINUM"]).set_index("DFCI_MRN")
v3 = pd.read_csv(V3_LABELS, sep="\t", usecols=["DFCI_MRN", "has_nepc", "has_avpc"]).set_index("DFCI_MRN")
j = agg_pl.join(v3, how="inner").dropna(subset=["has_nepc", "has_avpc"])
j["has_nepc"] = j["has_nepc"].astype(bool)
j["has_avpc"] = j["has_avpc"].astype(bool)
j["has_either"] = j["has_nepc"] | j["has_avpc"]

print(f"source: {V3_LABELS}")
print(f"joined: {len(j)} patients with both PLATINUM status and v3 label\n")

rows = []
for label, group in [("Platinum-treated", j[j["PLATINUM"] == 1]),
                     ("Platinum-naive",   j[j["PLATINUM"] == 0])]:
    n = len(group)
    rows.append({
        "group": label,
        "n": n,
        "NEPC_or_AVPC": fmt_pct(int(group["has_either"].sum()), n),
        "NEPC":         fmt_pct(int(group["has_nepc"].sum()),   n),
        "AVPC":         fmt_pct(int(group["has_avpc"].sum()),   n),
    })
pd.DataFrame(rows)

## 3. Univariate HRs (n_obs-adjusted) for testosterone and PSA

Abstract: *"Elevated mean testosterone was associated with faster time-to-platinum (HR 1.45 per SD at baseline, q=2×10⁻⁶; HR 1.40 at 3 months, q=8×10⁻⁴). Low mean PSA at 3 months was associated with slower time-to-platinum (HR 0.68 per SD, q=0.04)."*

In [ ]:
rows = []
for landmark in (0, 90):
    path = RUNS_DIR / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_univariate_nobs_adjusted.csv"
    if not path.exists():
        rows.append({"landmark_days": landmark, "lab": "-", "note": f"missing {path.name}"})
        continue
    df = pd.read_csv(path)
    for lab in ("Testosterone", "PSA"):
        sub = df[
            (df["lab_name"] == lab)
            & (df["feature_stat"] == "mean")
            & (df["endpoint"] == "platinum")
        ]
        if sub.empty:
            rows.append({"landmark_days": landmark, "lab": lab, "note": "row not found"})
            continue
        r = sub.iloc[0]
        rows.append({
            "landmark_days": int(r["landmark_days"]),
            "lab": lab,
            "feature": str(r["feature"]),
            "n_events": int(r["n_events_used"]),
            "HR_per_SD": float(r["hazard_ratio_per_sd"]),
            "ci_lo": float(r["ci_lower"]),
            "ci_hi": float(r["ci_upper"]),
            "p": float(r["p_value"]),
            "q": float(r["q_value"]),
        })
uni_df = pd.DataFrame(rows).sort_values(["lab", "landmark_days"]).reset_index(drop=True)
uni_df

## 4. Multivariable model performance — landmark_0, platinum, test set

Abstract: *"XGBoost on aggregated lab features achieved the highest mean AUC(t) (0.81), outperforming elastic-net Cox (0.72) and Dynamic-DeepHit (0.71)."*

In [ ]:
rows = []

cox_metrics = RUNS_DIR / "cox" / "landmark_0" / "both" / "cox_agg_multivariable_metrics.csv"
if cox_metrics.exists():
    df = pd.read_csv(cox_metrics)
    r = df[df["endpoint"] == "platinum"].iloc[0]
    rows.append({
        "model": "Elastic-net Cox", "n_test": int(r["n_test"]),
        "n_test_events": int(r["n_events_test"]),
        "c_index":     float(r["test_c_index"]),
        "mean_auc_t":  float(r["test_mean_auc_t"]),
        "ibs":         float(r["test_integrated_brier"]),
    })

xgb_metrics = RUNS_DIR / "xgboost" / "landmark_0" / "both" / "landmark_xgboost_metrics.csv"
if xgb_metrics.exists():
    df = pd.read_csv(xgb_metrics)
    r = df[df["endpoint"] == "platinum"].iloc[0]
    rows.append({
        "model": "XGBoost survival", "n_test": int(r["n_test"]),
        "n_test_events": int(r["n_test_events"]),
        "c_index":     float(r["c_index"]),
        "mean_auc_t":  float(r["mean_auc_t"]),
        "ibs":         float(r["integrated_brier"]),
    })

dh_metrics = RUNS_DIR / "deephit" / "landmark_0" / "PLATINUM" / "dynamic_deephit_metrics_PLATINUM.csv"
if dh_metrics.exists():
    df = pd.read_csv(dh_metrics)
    r = df.iloc[0]
    rows.append({
        "model": "Dynamic-DeepHit", "n_test": int(r["n_test"]),
        "n_test_events": int(r["n_test_events"]),
        "c_index":     float(r["c_index"]),
        "mean_auc_t":  float(r["mean_auc_t"]),
        "ibs":         float(r["integrated_brier"]),
    })

perf_df = pd.DataFrame(rows)
perf_df

## 5. CAIA site cohort sizes

Abstract: *"DFCI n=7,942/176 platinum; MSKCC n=15,557/98; JHU n=4,841/25"*

These come from external sites and are not derivable from PROFILE — edit the `SITE_COHORTS` dict in the Paths cell to update.

In [ ]:
site_df = (
    pd.DataFrame.from_dict(SITE_COHORTS, orient="index")
    .reset_index().rename(columns={"index": "site"})
)
site_df["pct_platinum"] = (100.0 * site_df["n_platinum"] / site_df["n"]).round(2)
site_df

## 6. Abstract-formatted summary

Renders the abstract sentences with the live numbers substituted in, so you can copy-paste straight into the abstract draft.

In [ ]:
def pct(num, denom):
    return f"{100.0 * num / denom:.0f}%"

treated = j[j["PLATINUM"] == 1]
naive   = j[j["PLATINUM"] == 0]

tt_either = pct(treated["has_either"].sum(), len(treated))
tt_nepc   = pct(treated["has_nepc"].sum(),   len(treated))
tt_avpc   = pct(treated["has_avpc"].sum(),   len(treated))
nv_either = pct(naive["has_either"].sum(),   len(naive))
nv_nepc   = pct(naive["has_nepc"].sum(),     len(naive))
nv_avpc   = pct(naive["has_avpc"].sum(),     len(naive))

def hr(landmark, lab):
    sub = uni_df[(uni_df["landmark_days"] == landmark) & (uni_df["lab"] == lab)]
    if sub.empty or "HR_per_SD" not in sub.columns or pd.isna(sub.iloc[0].get("HR_per_SD")):
        return "(missing)"
    r = sub.iloc[0]
    return f"HR {r['HR_per_SD']:.2f} per SD, q={r['q']:.2g}"

def auc(model_name):
    sub = perf_df[perf_df["model"] == model_name]
    return f"{sub.iloc[0]['mean_auc_t']:.2f}" if not sub.empty else "(missing)"

site_str = "; ".join(
    f"{s} n={SITE_COHORTS[s]['n']:,}/{SITE_COHORTS[s]['n_platinum']}"
    for s in ("DFCI", "MSKCC", "JHU")
)

summary = f"""\
Of {n_total:,} PROFILE patients, {n_platinum} ({100.0*n_platinum/n_total:.1f}%) initiated platinum.
NEPC/AVPC was identified in {tt_either} of platinum-treated patients (NEPC {tt_nepc}, AVPC {tt_avpc})
vs {nv_either} of platinum-naive (NEPC {nv_nepc}, AVPC {nv_avpc}).

Univariate, n_obs-adjusted (platinum endpoint, mean over pre-landmark window):
  Testosterone @ baseline    : {hr(0,  'Testosterone')}
  Testosterone @ 3 months    : {hr(90, 'Testosterone')}
  PSA          @ baseline    : {hr(0,  'PSA')}
  PSA          @ 3 months    : {hr(90, 'PSA')}

Multivariable time-to-platinum, landmark_0, test set mean AUC(t):
  XGBoost ........... {auc('XGBoost survival')}
  Elastic-net Cox ... {auc('Elastic-net Cox')}
  Dynamic-DeepHit ... {auc('Dynamic-DeepHit')}

CAIA site cohort sizes: {site_str}
"""
print(summary)